In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from aicsimageio import AICSImage
import time
import zarr
import os 
import sys
import czifile

### Do not change the code in the cell below

In [2]:
# This assumes that your notebook is inside the folder 'full movie', which is inside 'movie_data'
base_dir =  r'/Volumes/ActiveNAS/Valerie/TransferFolder/'

# Define the file directory and name
input_file_directory = '20250917_MALum/Optimization/'
save_file_directory = input_file_directory + 'zarr_file'
save_full_path = os.path.join(base_dir, save_file_directory)

zarr_directory = save_file_directory + '/all_channels_data'
zarr_full_path = os.path.join(base_dir, zarr_directory)



# Follow the instructions to properly run the notebook 
1. Save the your full movie in tif format to the following directory **LLSM-CME-ANALYSIS/Final/movie_data/full_movie/**
2. Movie must be in tif format 
3. Change the **input_file_name** below to match your movie name so it is loaded properly 
 
 **Nothing else needs to be changed**




In [3]:
######## Change name of movie file here #######
input_file_name = '20250917_MA3Sl1Ch2_FirstTimepoint-01_AcquisitionBlock13-Loop1-crop-dsdc.czi'
print('Your file name is: ', input_file_name)

Your file name is:  20250917_MA3Sl1Ch2_FirstTimepoint-01_AcquisitionBlock13-Loop1-crop-dsdc.czi


In [4]:
# Full path construction
input_full_path = os.path.join(base_dir, input_file_directory, input_file_name)

# Load the file
img = AICSImage(input_full_path)
# Check which reader is being used
print(type(img.reader))

<class 'aicsimageio.readers.czi_reader.CziReader'>


### Do not change the code in the cell below

In [5]:
dask_array = img.xarray_dask_data 
dask_array.name = 'all_channels_data'
# dask_array

In [6]:
# Check CZI metadata
print("Image shape:", img.shape)
print("Image dims:", img.dims)
print("Physical pixel sizes:", img.physical_pixel_sizes)
print("\nDask array info:")
print(dask_array.shape)
print(dask_array.dtype)
print("Chunk sizes:", dask_array.chunks)

Image shape: (205, 3, 157, 239, 343)
Image dims: <Dimensions [T: 205, C: 3, Z: 157, Y: 239, X: 343]>
Physical pixel sizes: PhysicalPixelSizes(Z=0.4, Y=0.14499219272808386, X=0.14499219272808386)

Dask array info:
(205, 3, 157, 239, 343)
uint16
Chunk sizes: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (1, 1, 1), (157,), (239,), (343,))


In [ ]:
# pip install czifile

Note: you may need to restart the kernel to use updated packages.


In [9]:
# Using a different import method in case dask-array isn't working
czi = czifile.CziFile(input_full_path)

# Create zarr store first
z_store = zarr.open(
    save_full_path, 
    mode='w',
    shape=(205, 3, 157, 239, 343),
    chunks=(1, 1, 5, 239, 343),
    dtype=np.uint16,
    compressor=zarr.Blosc(cname='lz4', clevel=5, shuffle=2)
)

# Write one timepoint at a time
for t in range(205):
    print(f"Processing timepoint {t+1}/205")
    # Read just this timepoint
    data_t = czi.asarray()[t:t+1].squeeze(axis=-1)  # Shape: (1, 3, 157, 239, 343)
    z_store[t] = data_t[0]  # Write this timepoint
    del data_t

print("Done!")

Processing timepoint 1/205
Processing timepoint 2/205
Processing timepoint 3/205
Processing timepoint 4/205
Processing timepoint 5/205
Processing timepoint 6/205
Processing timepoint 7/205
Processing timepoint 8/205
Processing timepoint 9/205
Processing timepoint 10/205
Processing timepoint 11/205
Processing timepoint 12/205
Processing timepoint 13/205
Processing timepoint 14/205
Processing timepoint 15/205
Processing timepoint 16/205
Processing timepoint 17/205
Processing timepoint 18/205
Processing timepoint 19/205
Processing timepoint 20/205
Processing timepoint 21/205
Processing timepoint 22/205
Processing timepoint 23/205
Processing timepoint 24/205
Processing timepoint 25/205
Processing timepoint 26/205
Processing timepoint 27/205
Processing timepoint 28/205
Processing timepoint 29/205
Processing timepoint 30/205
Processing timepoint 31/205
Processing timepoint 32/205
Processing timepoint 33/205
Processing timepoint 34/205
Processing timepoint 35/205
Processing timepoint 36/205
P

In [12]:
dask_array.attrs = []
dask_array.to_zarr(store = save_full_path, mode = 'w', compute = True)

MemoryError: std::bad_alloc

In [7]:
z2 = zarr.open(zarr_full_path, mode='r')
z2.info

Type,zarr.core.Array
Data type,uint16
Shape,"(204, 3, 143, 215, 319)"
Chunk shape,"(1, 1, 143, 215, 319)"
Order,C
Read-only,True
Compressor,"Blosc(cname='lz4', clevel=5, shuffle=SHUFFLE, blocksize=0)"
Store type,zarr.storage.DirectoryStore
No. bytes,12004569720 (11.2G)
No. bytes stored,1562036290 (1.5G)
Storage ratio,7.7


In [ ]:
# ### Valerie's extra code to make TIFFs for channel 3 directly from the .czi

# from tifffile import imwrite  # Optional, for saving TIFFs instead of zarrs

# # Output directory
# output_dir = "ch3"
# os.makedirs(output_dir, exist_ok=True)

# # Loop over timepoints and save each one
# for t in range(dask_array.sizes["T"]):
#     # Select timepoint t and channel 3 (adjust index if using a different channel, note that Python uses 0-based indexing)
#     volume = dask_array.isel(T=t, C=2).compute()  # Now a numpy array with shape (Z, Y, X)
    
#     # Convert to desired dtype if needed (e.g., uint16)
#     volume_np = volume.astype(np.uint16)
    
#     # Save to TIFF
#     imwrite(os.path.join(output_dir, f"test_ch3_t{t:03d}.tif"), volume_np)

# print(f"Saved {dask_array.sizes['T']} TIFFs to {output_dir}")

Saved 68 TIFFs to ch3
